# SciRet — Single-File Pipeline Notebook

One notebook, parameterized by `N_PAPERS`, runs the full SciRet pipeline end to end:
sample → chunk → embed (BGE-M3 + BM25) → retrieve (dense/BM25/hybrid) → rerank (MS MARCO cross-encoder) → generate (GPT-4o-mini) → evaluate (RAGAS).

**Replaces** the old `scale_1K/` ... `scale_100K/` folder-per-scale structure (5 notebooks each). Those are archived at
`6_legacy/notebooks_scale_1K_to_100K_archived_2026-08-05/` for reference, not deleted — git history and the archive both preserve them.

**Today's run (2026-08-05): 1K papers, full-text vs. title+abstract indexing pilot** — Phase 1B in `SciRet_Reboot_Plan.md`.
Set `RUN_FULLTEXT = True` below to build both conditions side by side and compare. This is the paper's first attempt at actually
solving the "index only titles/abstracts" limitation instead of just disclosing it (see `SciRet_Research_Approach.md` Section 1b/3.8).

To rerun at a different scale later: change `N_PAPERS` and `SCALE_LABEL` in the config cell. Nothing else in this notebook
should need to change — that's the whole point of consolidating to one file.


## 1 · Config
The only cell you should need to edit between runs.

In [ ]:
import os, sys, json, random, time
import numpy as np
import pandas as pd
import torch

# ── Scale (change these two for a different run) ────────────────────────────
SCALE_LABEL   = "scale_1K"
N_PAPERS      = 1000

# ── Full-text pilot toggle (Phase 1B) ────────────────────────────────────────
RUN_FULLTEXT           = True   # build a second, full-text-indexed condition alongside abstract-only
REQUIRE_FULLTEXT_FOR_SAMPLE = True   # if True, only sample papers that actually have full text available,
                                     # so the "full-text" condition never silently falls back to abstract-only

# ── Expensive-step toggles ───────────────────────────────────────────────────
RUN_GENERATION_RAGAS = True   # calls the OpenAI API (gpt-4o-mini) — costs money/time; set False to skip and just get retrieval/rerank numbers

# ── Fixed pipeline parameters — do not change between runs, these are what make scales comparable ──
RANDOM_SEED   = 42
CHUNK_SIZE    = 400     # tokens
CHUNK_OVERLAP = 50      # tokens
RRF_K         = 60
TOP_K_STAGE1  = 50      # candidates handed to the reranker
EVAL_K_VALUES = [1, 3, 5, 10, 20]   # Recall@K
P_K_VALUES    = [1, 3, 5, 10]       # Precision@K (reranking)
GEN_MODEL     = "gpt-4o-mini"
JUDGE_MODEL   = "gpt-4o-mini"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

# ── Paths ─────────────────────────────────────────────────────────────────────
# Local repo layout by default. On Kaggle, override RAW_DIR to your input dataset mount,
# e.g. RAW_DIR = "/kaggle/input/CORD-19-research-challenge/2022-06-02"
BASE_DIR         = os.path.abspath("..")
DATA_ROOT        = os.path.join(BASE_DIR, "1_data")
RAW_DIR          = os.path.join(DATA_ROOT, "raw")
DOCUMENT_PARSES_DIR = os.path.join(RAW_DIR, "document_parses")   # expected subfolders: pdf_json/, pmc_json/
RESULTS_DIR       = os.path.join(BASE_DIR, "4_results", SCALE_LABEL)
GT_PATH           = os.path.join(DATA_ROOT, "eval", "ground_truth.json")
QUERIES_PATH      = os.path.join(DATA_ROOT, "eval", "queries.json")
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"Scale: {SCALE_LABEL} | N={N_PAPERS:,} | Device: {DEVICE} | Full-text pilot: {RUN_FULLTEXT}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print(f"Results dir: {RESULTS_DIR}")


## 2 · Dependencies

In [ ]:
import subprocess, sys as _sys
_pkgs = [
    "sentence-transformers", "rank_bm25", "chromadb",
    "openai", "python-dotenv",
    "ragas==0.2.6", "langchain-openai", "langchain-huggingface",
]
subprocess.check_call([_sys.executable, "-m", "pip", "install", "-q", *_pkgs])
print("Dependencies ready.")


## 3 · Load & sample CORD-19 metadata
Same stratified-by-year sampling as the archived scale notebooks (seed 42), so this run stays comparable
to the previously reported 1K abstract-only numbers.

In [ ]:
meta_path = os.path.join(RAW_DIR, "metadata.csv")
df_meta = pd.read_csv(meta_path, low_memory=False)
df_meta = df_meta[df_meta["abstract"].notna()].copy()
df_meta = df_meta.drop_duplicates(subset=["cord_uid"])
print(f"Papers with abstract (deduped): {len(df_meta):,}")

# ── Full-text availability (Phase 1B) ────────────────────────────────────────
# CORD-19's 2022-06-02 release marks availability via path columns (pdf_json_files / pmc_json_files,
# semicolon-separated relative paths, NaN if absent). Older releases used boolean has_pdf_parse /
# has_pmc_xml_parse columns instead. Detect whichever this metadata.csv actually has.
if "pdf_json_files" in df_meta.columns or "pmc_json_files" in df_meta.columns:
    _fulltext_mask = df_meta.get("pdf_json_files", pd.Series(index=df_meta.index)).notna() | \
                     df_meta.get("pmc_json_files", pd.Series(index=df_meta.index)).notna()
    print("Full-text availability column style: path columns (pdf_json_files / pmc_json_files)")
elif "has_pdf_parse" in df_meta.columns or "has_pmc_xml_parse" in df_meta.columns:
    _fulltext_mask = df_meta.get("has_pdf_parse", False).astype(bool) | \
                      df_meta.get("has_pmc_xml_parse", False).astype(bool)
    print("Full-text availability column style: boolean columns (has_pdf_parse / has_pmc_xml_parse)")
else:
    _fulltext_mask = pd.Series(False, index=df_meta.index)
    print("WARNING: no recognized full-text-availability column found in metadata.csv — "
          "full-text coverage will read as 0%. Check the CORD-19 release/columns.")

df_meta["has_fulltext"] = _fulltext_mask
coverage_pct = 100 * df_meta["has_fulltext"].mean()
print(f"Full-text coverage in full metadata: {df_meta['has_fulltext'].sum():,} / {len(df_meta):,} papers ({coverage_pct:.1f}%)")

if RUN_FULLTEXT and REQUIRE_FULLTEXT_FOR_SAMPLE:
    sample_pool = df_meta[df_meta["has_fulltext"]].copy()
    print(f"REQUIRE_FULLTEXT_FOR_SAMPLE=True — sampling only from the {len(sample_pool):,} full-text-available papers, "
          f"so the abstract-only and full-text conditions use the EXACT SAME {N_PAPERS} papers.")
else:
    sample_pool = df_meta.copy()
    print(f"Sampling from all {len(sample_pool):,} papers with an abstract (full-text condition may fall back "
          f"to abstract-only for papers lacking full text — check the coverage report below).")

assert len(sample_pool) >= N_PAPERS, (
    f"Only {len(sample_pool):,} candidate papers available, need {N_PAPERS:,}. "
    f"Set REQUIRE_FULLTEXT_FOR_SAMPLE=False to sample from the full abstract pool instead."
)


In [ ]:
# Stratified sample by publication year (same method as the archived notebooks)
sample_pool["year"] = pd.to_datetime(sample_pool["publish_time"], errors="coerce").dt.year
sample_pool = sample_pool[sample_pool["year"].notna()].copy()

year_counts = sample_pool["year"].value_counts()
year_weights = year_counts / year_counts.sum()

sampled_ids = []
for year, weight in year_weights.items():
    n_year = max(1, int(round(N_PAPERS * weight)))
    pool = sample_pool[sample_pool["year"] == year]["cord_uid"].tolist()
    n_year = min(n_year, len(pool))
    sampled_ids.extend(random.sample(pool, n_year))

random.shuffle(sampled_ids)
sampled_ids = sampled_ids[:N_PAPERS]
df_sample = sample_pool[sample_pool["cord_uid"].isin(sampled_ids)].reset_index(drop=True)
print(f"Sampled: {len(df_sample):,} papers (target {N_PAPERS:,})")
print(f"Full-text available within sample: {df_sample['has_fulltext'].sum():,} / {len(df_sample):,} "
      f"({100*df_sample['has_fulltext'].mean():.1f}%)")


## 4 · Full-text loader (Phase 1B — new)
Uses CORD-19's own pre-parsed `document_parses/*_json/*.json` files (each has a `body_text` list of paragraph
objects) rather than parsing raw PDFs — much less work, and it's what CORD-19 already ships.

In [ ]:
def _first_existing_path(rel_paths_str, base_dir):
    """pdf_json_files / pmc_json_files can list multiple semicolon-separated candidate paths."""
    if not isinstance(rel_paths_str, str) or not rel_paths_str.strip():
        return None
    for rel in rel_paths_str.split(";"):
        rel = rel.strip()
        if not rel:
            continue
        candidate = os.path.join(base_dir, rel)
        if os.path.exists(candidate):
            return candidate
    return None


def load_body_text(row, raw_dir=RAW_DIR) -> str:
    """Return concatenated body_text for a metadata row, or '' if no full-text JSON is found/parseable."""
    for col in ("pdf_json_files", "pmc_json_files"):
        if col not in row.index:
            continue
        path = _first_existing_path(row.get(col), raw_dir)
        if path is None:
            continue
        try:
            with open(path, "r", encoding="utf-8") as f:
                doc = json.load(f)
            paragraphs = [seg.get("text", "") for seg in doc.get("body_text", [])]
            text = " ".join(p for p in paragraphs if p).strip()
            if text:
                return text
        except (OSError, json.JSONDecodeError) as e:
            print(f"  WARNING: failed to parse {path}: {e}")
            continue
    return ""


if RUN_FULLTEXT:
    print("Loading full-text bodies for the sample (this reads one JSON file per paper)...")
    t0 = time.time()
    body_texts = []
    n_found = 0
    for _, row in df_sample.iterrows():
        bt = load_body_text(row)
        if bt:
            n_found += 1
        body_texts.append(bt)
    df_sample["body_text"] = body_texts
    print(f"Full text loaded for {n_found:,} / {len(df_sample):,} sampled papers "
          f"({100*n_found/len(df_sample):.1f}%) in {time.time()-t0:.1f}s")
    if n_found < len(df_sample):
        print(f"  {len(df_sample) - n_found} papers had a metadata full-text pointer but no readable JSON on disk — "
              f"check DOCUMENT_PARSES_DIR / RAW_DIR matches where the CORD-19 document_parses folder actually is.")


## 5 · Chunking (sentence-window — same strategy as Table `tab:chunking`, now applied to full text too)

In [ ]:
import re

def sentence_window_chunk(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    chunks, current, current_len = [], [], 0
    for sent in sentences:
        tokens = sent.split()
        if current_len + len(tokens) > chunk_size and current:
            chunks.append(" ".join(current))
            overlap_sents, overlap_len = [], 0
            for s in reversed(current):
                l = len(s.split())
                if overlap_len + l <= overlap:
                    overlap_sents.insert(0, s)
                    overlap_len += l
                else:
                    break
            current, current_len = overlap_sents, overlap_len
        current.append(sent)
        current_len += len(tokens)
    if current:
        chunks.append(" ".join(current))
    return chunks


def build_chunks(df, text_fn, label):
    rows = []
    for _, paper in df.iterrows():
        text = text_fn(paper)
        if not text or not text.strip():
            continue
        for i, chunk in enumerate(sentence_window_chunk(text)):
            rows.append({
                "cord_uid": paper["cord_uid"],
                "chunk_id": f"{paper['cord_uid']}_{label}_{i}",
                "text": chunk,
                "title": paper.get("title", ""),
                "year": paper.get("year", None),
                "token_count": len(chunk.split()),
            })
    df_out = pd.DataFrame(rows)
    print(f"[{label}] Chunks: {len(df_out):,} from {df['cord_uid'].nunique():,} papers | "
          f"mean tokens: {df_out['token_count'].mean():.1f}" if len(df_out) else f"[{label}] 0 chunks produced")
    return df_out


CONDITIONS = {}

df_chunks_abstract = build_chunks(
    df_sample, lambda p: f"{p.get('title','')} {p.get('abstract','')}".strip(), "abstract"
)
df_chunks_abstract.to_parquet(os.path.join(RESULTS_DIR, "chunks_abstract.parquet"), index=False)
CONDITIONS["abstract"] = df_chunks_abstract

if RUN_FULLTEXT:
    # papers with no body_text fall back to title+abstract so every paper still contributes at least one chunk
    def _fulltext_or_fallback(p):
        bt = p.get("body_text", "")
        if bt and bt.strip():
            return f"{p.get('title','')} {bt}".strip()
        return f"{p.get('title','')} {p.get('abstract','')}".strip()

    df_chunks_fulltext = build_chunks(df_sample, _fulltext_or_fallback, "fulltext")
    df_chunks_fulltext.to_parquet(os.path.join(RESULTS_DIR, "chunks_fulltext.parquet"), index=False)
    CONDITIONS["fulltext"] = df_chunks_fulltext


## 6 · Embedding + indexing (BGE-M3 dense + BM25) — built once per condition

In [ ]:
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import pickle

print("Loading BGE-M3...")
embed_model = SentenceTransformer("BAAI/bge-m3", device=DEVICE)
BATCH_SIZE = 64 if DEVICE == "cuda" else 8

INDEXES = {}  # condition -> dict(chunk_ids, texts, cord_uids, embeddings, bm25)

for cond_name, df_chunks in CONDITIONS.items():
    print(f"\n[{cond_name}] Embedding {len(df_chunks):,} chunks...")
    texts = df_chunks["text"].tolist()
    chunk_ids = df_chunks["chunk_id"].tolist()
    cord_uids = df_chunks["cord_uid"].tolist()

    t0 = time.time()
    embeddings = embed_model.encode(
        texts, batch_size=BATCH_SIZE, show_progress_bar=True, normalize_embeddings=True
    )
    print(f"[{cond_name}] Dense embedding done in {(time.time()-t0)/60:.1f} min | shape {embeddings.shape}")
    np.save(os.path.join(RESULTS_DIR, f"bge_m3_embeddings_{cond_name}.npy"), embeddings)

    tokenized = [t.lower().split() for t in texts]
    bm25 = BM25Okapi(tokenized)
    with open(os.path.join(RESULTS_DIR, f"bm25_index_{cond_name}.pkl"), "wb") as f:
        pickle.dump(bm25, f)

    INDEXES[cond_name] = {
        "chunk_ids": chunk_ids,
        "texts": texts,
        "id_to_text": dict(zip(chunk_ids, texts)),
        "id_to_cord": dict(zip(chunk_ids, cord_uids)),
        "embeddings": embeddings,
        "bm25": bm25,
    }
    print(f"[{cond_name}] Index ready: {len(chunk_ids):,} chunks")


## 7 · Evaluation queries + ground truth
Uses the 50-query stratified set (`1_data/eval/queries.json`) — see the Phase 1 note in `SciRet_Reboot_Plan.md`
about reconciling this with the previously-reported 15-query numbers before treating these results as final.
Ground truth: loads `ground_truth.json` if present; otherwise falls back to interim pseudo-labels (top-3 hybrid
on the abstract condition — matching the paper's documented methodology), and reuses the same labels for the
full-text condition since ground truth is defined at the paper (`cord_uid`) level, independent of how each
paper happens to be indexed.

In [ ]:
with open(QUERIES_PATH) as f:
    QUERIES = json.load(f)
print(f"Queries: {len(QUERIES)}")

def hybrid_retrieve_cordids(query, cond_name, k=3):
    idx = INDEXES[cond_name]
    q_emb = embed_model.encode([query], normalize_embeddings=True)
    sims = (idx["embeddings"] @ q_emb[0])
    dense_top = list(np.argsort(sims)[::-1][:TOP_K_STAGE1])
    tokens = query.lower().split()
    bm25_top = list(np.argsort(idx["bm25"].get_scores(tokens))[::-1][:TOP_K_STAGE1])
    rrf = {}
    for lst in [dense_top, bm25_top]:
        for rank, i in enumerate(lst):
            cid = idx["chunk_ids"][i]
            rrf[cid] = rrf.get(cid, 0) + 1.0 / (RRF_K + rank + 1)
    top = sorted(rrf, key=rrf.get, reverse=True)[:k]
    return [idx["id_to_cord"][c] for c in top]

if os.path.exists(GT_PATH):
    with open(GT_PATH) as f:
        GROUND_TRUTH = json.load(f)
    GT_SOURCE = "manual/external file"
else:
    print("ground_truth.json not found — building INTERIM pseudo-labels from abstract-condition hybrid top-3.")
    print("This carries the same circularity caveat as the paper's current Limitations/Section 3.4 text.")
    GROUND_TRUTH = {q: hybrid_retrieve_cordids(q, "abstract", k=3) for q in QUERIES}
    GT_SOURCE = "pseudo-labels (abstract-condition hybrid top-3, this run)"
    with open(os.path.join(RESULTS_DIR, "ground_truth_pseudo.json"), "w") as f:
        json.dump(GROUND_TRUTH, f, indent=2)

print(f"Ground truth source: {GT_SOURCE} | {len(GROUND_TRUTH)} queries labeled")


## 8 · Retrieval evaluation — Recall@K, per condition

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def dense_retrieve(query, cond_name, k=50):
    idx = INDEXES[cond_name]
    q_emb = embed_model.encode([query], normalize_embeddings=True)
    sims = cosine_similarity(q_emb, idx["embeddings"])[0]
    top = np.argsort(sims)[::-1][:k]
    return [idx["chunk_ids"][i] for i in top]

def bm25_retrieve(query, cond_name, k=50):
    idx = INDEXES[cond_name]
    scores = idx["bm25"].get_scores(query.lower().split())
    top = np.argsort(scores)[::-1][:k]
    return [idx["chunk_ids"][i] for i in top]

def rrf_fuse(lists, k=RRF_K):
    scores = {}
    for lst in lists:
        for rank, doc_id in enumerate(lst):
            scores[doc_id] = scores.get(doc_id, 0) + 1.0 / (k + rank + 1)
    return sorted(scores, key=scores.get, reverse=True)

def hybrid_retrieve(query, cond_name, k=50):
    return rrf_fuse([dense_retrieve(query, cond_name, k), bm25_retrieve(query, cond_name, k)])[:k]

RETRIEVERS = {"dense": dense_retrieve, "bm25": bm25_retrieve, "hybrid": hybrid_retrieve}

recall_summary_rows = []
for cond_name in CONDITIONS:
    idx = INDEXES[cond_name]
    recall_results = {s: {k: [] for k in EVAL_K_VALUES} for s in RETRIEVERS}
    for query in QUERIES:
        if query not in GROUND_TRUTH or not GROUND_TRUTH[query]:
            continue
        relevant = set(GROUND_TRUTH[query])
        for sys_name, fn in RETRIEVERS.items():
            ranked_ids = fn(query, cond_name, k=max(EVAL_K_VALUES))
            for k in EVAL_K_VALUES:
                top_k_cords = {idx["id_to_cord"].get(c, "") for c in ranked_ids[:k]}
                recall_results[sys_name][k].append(len(top_k_cords & relevant) / len(relevant))
    print(f"\nRecall@K — {cond_name} ({len(df_sample):,} papers, {len(idx['chunk_ids']):,} chunks)")
    print(f"{'K':>4}  {'Dense':>8}  {'BM25':>8}  {'Hybrid':>8}")
    for k in EVAL_K_VALUES:
        row = {"condition": cond_name, "k": k}
        for sys_name in RETRIEVERS:
            row[sys_name] = float(np.mean(recall_results[sys_name][k])) if recall_results[sys_name][k] else float("nan")
        print(f"{k:>4}  {row['dense']:>8.3f}  {row['bm25']:>8.3f}  {row['hybrid']:>8.3f}")
        recall_summary_rows.append(row)

df_recall = pd.DataFrame(recall_summary_rows)
df_recall.to_csv(os.path.join(RESULTS_DIR, "recall_at_k_comparison.csv"), index=False)
print(f"\nSaved: {RESULTS_DIR}/recall_at_k_comparison.csv")


## 9 · Reranking — Precision@K before/after MS MARCO cross-encoder, per condition

In [ ]:
from sentence_transformers import CrossEncoder

print("Loading cross-encoder (ms-marco-MiniLM-L-6-v2)...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, cond_name, candidate_ids):
    idx = INDEXES[cond_name]
    pairs = [(query, idx["id_to_text"][cid]) for cid in candidate_ids if cid in idx["id_to_text"]]
    if not pairs:
        return []
    scores = reranker.predict(pairs)
    ranked = sorted(zip(candidate_ids[:len(pairs)], scores), key=lambda x: x[1], reverse=True)
    return [cid for cid, _ in ranked]

rerank_summary_rows = []
for cond_name in CONDITIONS:
    idx = INDEXES[cond_name]
    p_no_rerank = {k: [] for k in P_K_VALUES}
    p_rerank = {k: [] for k in P_K_VALUES}
    for query in QUERIES:
        if query not in GROUND_TRUTH or not GROUND_TRUTH[query]:
            continue
        relevant = set(GROUND_TRUTH[query])
        candidates = hybrid_retrieve(query, cond_name, k=TOP_K_STAGE1)
        reranked = rerank(query, cond_name, candidates)
        for k in P_K_VALUES:
            top_k_no = {idx["id_to_cord"].get(c, "") for c in candidates[:k]}
            top_k_re = {idx["id_to_cord"].get(c, "") for c in reranked[:k]}
            p_no_rerank[k].append(len(top_k_no & relevant) / k)
            p_rerank[k].append(len(top_k_re & relevant) / k)
    print(f"\nReranking impact — {cond_name}")
    print(f"{'K':>4}  {'No rerank':>10}  {'Reranked':>10}  {'Delta':>8}")
    for k in P_K_VALUES:
        nr, rr = float(np.mean(p_no_rerank[k])), float(np.mean(p_rerank[k]))
        print(f"{k:>4}  {nr:>10.3f}  {rr:>10.3f}  {rr-nr:>+8.3f}")
        rerank_summary_rows.append({"condition": cond_name, "k": k, "p_no_rerank": nr, "p_rerank": rr, "delta": rr - nr})

df_rerank = pd.DataFrame(rerank_summary_rows)
df_rerank.to_csv(os.path.join(RESULTS_DIR, "reranking_precision_comparison.csv"), index=False)
print(f"\nSaved: {RESULTS_DIR}/reranking_precision_comparison.csv")


## 10 · Generation + RAGAS (optional — costs OpenAI API calls)
Set `RUN_GENERATION_RAGAS = False` in the config cell to skip this and stop after retrieval/reranking numbers.
Requires `OPENAI_API_KEY` in a `.env` file at the project root.

In [ ]:
if RUN_GENERATION_RAGAS:
    from dotenv import load_dotenv
    load_dotenv(os.path.join(BASE_DIR, ".env"))
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
    assert OPENAI_API_KEY, "OPENAI_API_KEY not found in .env — set it or set RUN_GENERATION_RAGAS = False"

    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)

    def generate_answer(query, context_chunks):
        context = "\n\n".join(f"[{i+1}] {c}" for i, c in enumerate(context_chunks))
        prompt = (
            "You are a scientific literature assistant. "
            "Answer the question using ONLY the provided context. "
            "Cite source numbers [1], [2], etc. for each claim.\n\n"
            f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer:"
        )
        try:
            resp = client.chat.completions.create(
                model=GEN_MODEL, messages=[{"role": "user", "content": prompt}],
                max_tokens=2048, temperature=0.1,
            )
            return resp.choices[0].message.content
        except Exception as e:
            return f"ERROR: {e}"

    def hybrid_retrieve_top5_text(query, cond_name):
        idx = INDEXES[cond_name]
        top5 = hybrid_retrieve(query, cond_name, k=5)
        return [idx["id_to_text"][c] for c in top5 if c in idx["id_to_text"]]

    eval_runs_by_condition = {}
    for cond_name in CONDITIONS:
        print(f"\n[{cond_name}] Generating answers for {len(QUERIES)} queries...")
        records = []
        for i, query in enumerate(QUERIES):
            contexts = hybrid_retrieve_top5_text(query, cond_name)
            answer = generate_answer(query, contexts)
            records.append({"question": query, "answer": answer, "contexts": contexts, "ground_truth": query})
            if (i + 1) % 10 == 0:
                print(f"  {i+1}/{len(QUERIES)} done")
        df_eval = pd.DataFrame(records)
        df_eval.to_parquet(os.path.join(RESULTS_DIR, f"eval_runs_{cond_name}.parquet"), index=False)
        errors = sum(1 for r in records if str(r["answer"]).startswith("ERROR:"))
        print(f"[{cond_name}] Saved {len(records)} answers ({errors} errors) -> eval_runs_{cond_name}.parquet")
        eval_runs_by_condition[cond_name] = df_eval
else:
    print("RUN_GENERATION_RAGAS = False — skipping generation and RAGAS.")


In [ ]:
if RUN_GENERATION_RAGAS:
    from datasets import Dataset
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
    from ragas.llms import LangchainLLMWrapper
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from ragas.run_config import RunConfig
    from langchain_openai import ChatOpenAI
    from langchain_huggingface import HuggingFaceEmbeddings

    ragas_llm = LangchainLLMWrapper(
        ChatOpenAI(model=JUDGE_MODEL, api_key=OPENAI_API_KEY, temperature=0, max_tokens=512)
    )
    ragas_emb = LangchainEmbeddingsWrapper(HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"))
    run_config = RunConfig(timeout=120, max_retries=5, max_workers=2)

    ragas_summary_rows = []
    for cond_name, df_eval in eval_runs_by_condition.items():
        df_valid = df_eval[~df_eval["answer"].astype(str).str.startswith("ERROR:")].reset_index(drop=True)
        if len(df_valid) < len(QUERIES) * 0.8:
            print(f"[{cond_name}] Only {len(df_valid)}/{len(QUERIES)} valid answers — check API errors before trusting RAGAS here.")
        ds = Dataset.from_dict({
            "question": df_valid["question"].tolist(),
            "answer": df_valid["answer"].tolist(),
            "contexts": df_valid["contexts"].tolist(),
            "ground_truth": df_valid["ground_truth"].tolist(),
        })
        print(f"\n[{cond_name}] Running RAGAS on {len(ds)} samples...")
        result = evaluate(
            ds, metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
            llm=ragas_llm, embeddings=ragas_emb, run_config=run_config,
        )
        df_scores = result.to_pandas()
        df_scores.to_csv(os.path.join(RESULTS_DIR, f"ragas_scores_{cond_name}.csv"), index=False)
        agg = df_scores.select_dtypes(include=["number"]).mean().to_dict()
        agg_row = {"condition": cond_name, **agg}
        ragas_summary_rows.append(agg_row)
        print(f"[{cond_name}] {agg}")

    df_ragas_summary = pd.DataFrame(ragas_summary_rows)
    df_ragas_summary.to_csv(os.path.join(RESULTS_DIR, "ragas_comparison.csv"), index=False)
    print(f"\nSaved: {RESULTS_DIR}/ragas_comparison.csv")


## 11 · Full-text vs. abstract-only — decision summary
Mirrors the Phase 1B decision gate in `SciRet_Reboot_Plan.md`: does full-text indexing meaningfully change
the story? Read this alongside the printed tables above before deciding whether to extend full-text to 5K/15K.

In [ ]:
print("=" * 70)
print(f"SUMMARY — {SCALE_LABEL} ({N_PAPERS:,} papers), ground truth: {GT_SOURCE}")
print("=" * 70)

print("\nRecall@K comparison:")
print(df_recall.pivot(index="k", columns="condition", values="hybrid").round(3))

print("\nReranking Precision@K comparison (delta = reranked - no_rerank):")
print(df_rerank.pivot(index="k", columns="condition", values="delta").round(3))

if RUN_GENERATION_RAGAS:
    print("\nRAGAS comparison:")
    print(df_ragas_summary.set_index("condition").round(3))

print("""
\nNext step: compare the numbers above against the currently reported abstract-only 1K numbers in main.tex
(Recall@10=1.000, P@5 0.600->0.404, RAGAS faithfulness=0.917). If full-text changes the story meaningfully,
see the Phase 1B decision gate in SciRet_Reboot_Plan.md for what to do next (extend to 5K/15K vs. keep as a
1K depth ablation). If not, the Limitations section should still be updated to say full-text WAS tested at 1K,
not left as an untested disclosure.
""")
